# WC 2026 Predictions — Hybrid ELO + Pi-Ratings GLM

Generates score predictions for all unplayed WC 2026 group-stage matches using the best validated model (Exp D, 4.387 pts/match LOTO-CV).

**Model:** Poisson GLM with hybrid ELO + pi-ratings features, retrained on the full training set (all matches up to 2026-06-01).  
**Score strategy:** predict (h, a) that maximises expected Sporza points under the Poisson joint distribution.  
**Live update:** played WC 2026 results from `wc2026_fixtures.parquet` are appended to the match history before computing pi-ratings and form features, so each fixture is predicted with up-to-date context.  
**Output:** `data/predictions/wc2026_predictions.csv`

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
from functools import lru_cache
from pathlib import Path
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from penaltyblog.ratings import PiRatingSystem

DATA    = Path('../../data/processed')
INTERIM = Path('../../data/interim')
OUT_DIR = Path('../../data/predictions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

FORM_WINDOW = 5
MAX_GOALS   = 8

# Exact feature order used in Exp D (notebook 19)
HOME_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'pi_home', 'pi_away', 'pi_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
AWAY_FEATURES = [
    'away_elo', 'home_elo', 'elo_diff',
    'pi_away', 'pi_home', 'pi_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]

print('Paths OK.')
print(f'Output directory: {OUT_DIR.resolve()}')

Paths OK.
Output directory: /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/predictions


## 1. Load data

In [2]:
# Full training set (all matches before WC 2026)
train = pd.read_parquet(DATA / 'train.parquet')
print(f'Training set: {len(train)} matches  ({train.date.min().date()} → {train.date.max().date()})')

# Full match history for pi-rating and form computation
matches_full = pd.read_parquet(INTERIM / 'matches.parquet')
print(f'Full match history: {len(matches_full)} matches  ({matches_full.date.min().date()} → {matches_full.date.max().date()})')

# WC 2026 fixtures — source of truth for played results and upcoming schedule
fixtures_all = pd.read_parquet(INTERIM / 'wc2026_fixtures.parquet')
# Exclude knockout placeholders (team names like '2A', '1F')
is_placeholder = fixtures_all['home_team'].str.match(r'^[0-9]') | fixtures_all['away_team'].str.match(r'^[0-9]')
fixtures_all = fixtures_all[~is_placeholder].copy()

played_wc   = fixtures_all[fixtures_all['played'] == True].copy()
unplayed_wc = fixtures_all[fixtures_all['played'] == False].copy()
print(f'\nWC 2026 played:   {len(played_wc)}')
print(f'WC 2026 unplayed: {len(unplayed_wc)}')
if len(played_wc):
    print()
    print('Played results:')
    print(played_wc[['date', 'home_team', 'away_team', 'home_score', 'away_score']].to_string(index=False))

Training set: 11106 matches  (2010-01-02 → 2022-10-30)
Full match history: 32288 matches  (1990-01-12 → 2026-06-11)

WC 2026 played:   2
WC 2026 unplayed: 70

Played results:
      date   home_team      away_team  home_score  away_score
2026-06-11      Mexico   South Africa         2.0         0.0
2026-06-11 South Korea Czech Republic         2.0         1.0


## 2. Extend match history with played WC 2026 results

Append played WC matches to the historical record so that pi-ratings and rolling form for
later fixtures reflect what has actually happened in the tournament.

In [3]:
if len(played_wc) > 0:
    wc_results = played_wc[['date', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
    wc_results['tournament'] = 'FIFA World Cup'
    wc_results['city']       = ''
    wc_results['country']    = ''
    wc_results['neutral']    = True
    wc_results['home_score'] = wc_results['home_score'].astype(int)
    wc_results['away_score'] = wc_results['away_score'].astype(int)

    # Avoid double-counting matches already in matches_full (in case the file was refreshed)
    existing_keys = set(zip(matches_full['date'].astype(str), matches_full['home_team'], matches_full['away_team']))
    new_results = wc_results[
        ~wc_results.apply(lambda r: (str(r['date'])[:10], r['home_team'], r['away_team']) in existing_keys, axis=1)
    ]

    matches_live = pd.concat([matches_full, new_results], ignore_index=True).sort_values('date').reset_index(drop=True)
    print(f'Appended {len(new_results)} new WC result(s) to match history.')
else:
    matches_live = matches_full.copy()
    print('No played WC results to append yet.')

print(f'Live match history: {len(matches_live)} matches  (up to {matches_live.date.max().date()})')

Appended 0 new WC result(s) to match history.
Live match history: 32288 matches  (up to 2026-06-11)


## 3. Compute per-fixture pi-rating snapshots

Pi-ratings are updated incrementally — each fixture uses all matches strictly before its
date, so a team's WC result feeds into the rating for their next game.

In [4]:
# One pi-rating snapshot per unique prediction date (multiple fixtures share the same matchday)
prediction_dates = sorted(unplayed_wc['date'].unique())
print(f'{len(prediction_dates)} prediction dates: {[str(d)[:10] for d in prediction_dates]}')

live_sorted = matches_live.sort_values('date').reset_index(drop=True)

pi_snapshots = {}  # date -> team_ratings dict
for pred_date in prediction_dates:
    pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)
    training_slice = live_sorted[live_sorted['date'] < pred_date]
    for _, row in training_slice.iterrows():
        pi.update_ratings(row['home_team'], row['away_team'], int(row['home_score'] - row['away_score']))
    pi_snapshots[pred_date] = pi.team_ratings
    print(f'  {str(pred_date)[:10]}: {len(training_slice)} matches → {len(pi.team_ratings)} teams rated')

print('\nPi-rating snapshots computed.')

def apply_pi_features(df, snapshots):
    df = df.copy()
    df['pi_home'] = df.apply(lambda r: snapshots[r['date']].get(r['home_team'], {}).get('home', 0.0), axis=1)
    df['pi_away'] = df.apply(lambda r: snapshots[r['date']].get(r['away_team'], {}).get('away',  0.0), axis=1)
    df['pi_diff'] = df['pi_home'] - df['pi_away']
    return df

unplayed_wc = apply_pi_features(unplayed_wc, pi_snapshots)

missing = unplayed_wc[(unplayed_wc['pi_home'] == 0.0) | (unplayed_wc['pi_away'] == 0.0)]
if len(missing):
    print('Teams defaulting to pi=0.0:', set(missing['home_team']) | set(missing['away_team']))
else:
    print('All teams have pi-ratings for their fixture date.')

# Training set uses pre-tournament ratings (snapshot for the first prediction date)
pre_tourney = pi_snapshots[prediction_dates[0]]
train = train.copy()
train['pi_home'] = train['home_team'].map(lambda t: pre_tourney.get(t, {}).get('home', 0.0))
train['pi_away'] = train['away_team'].map(lambda t: pre_tourney.get(t, {}).get('away',  0.0))
train['pi_diff'] = train['pi_home'] - train['pi_away']
print(f'Pi features added to training set ({len(train)} matches).')

16 prediction dates: ['2026-06-12', '2026-06-13', '2026-06-14', '2026-06-15', '2026-06-16', '2026-06-17', '2026-06-18', '2026-06-19', '2026-06-20', '2026-06-21', '2026-06-22', '2026-06-23', '2026-06-24', '2026-06-25', '2026-06-26', '2026-06-27']


  2026-06-12: 32288 matches → 326 teams rated


  2026-06-13: 32288 matches → 326 teams rated


  2026-06-14: 32288 matches → 326 teams rated


  2026-06-15: 32288 matches → 326 teams rated


...

  2026-06-23: 32288 matches → 326 teams rated


  2026-06-24: 32288 matches → 326 teams rated


  2026-06-25: 32288 matches → 326 teams rated


  2026-06-26: 32288 matches → 326 teams rated


  2026-06-27: 32288 matches → 326 teams rated

Pi-rating snapshots computed.
All teams have pi-ratings for their fixture date.
Pi features added to training set (11106 matches).


## 4. Compute per-fixture rolling form features

Same window-5 rolling form as the training pipeline, but computed as-of each fixture date
so that played WC results shift the form of teams going into their next match.

In [5]:
def build_team_view(matches_df):
    home = matches_df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home.columns = ['date', 'team', 'goals_scored', 'goals_conceded']
    away = matches_df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away.columns = ['date', 'team', 'goals_scored', 'goals_conceded']
    tv = pd.concat([home, away]).sort_values(['team', 'date']).reset_index(drop=True)
    tv['win'] = (tv['goals_scored'] > tv['goals_conceded']).astype(float)
    return tv


def compute_form_snapshots(matches_df, prediction_dates, n=FORM_WINDOW):
    """Return dict: date -> {team: {form_wr, form_gf, form_ga}} using all matches before that date."""
    tv_full = build_team_view(matches_df)
    snapshots = {}
    for pred_date in prediction_dates:
        tv = tv_full[tv_full['date'] < pred_date].copy()
        grp = tv.groupby('team')
        tv['form_wr'] = grp['win'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        tv['form_gf'] = grp['goals_scored'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        tv['form_ga'] = grp['goals_conceded'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        latest = (
            tv.sort_values('date')
            .drop_duplicates(subset='team', keep='last')
            .set_index('team')[['form_wr', 'form_gf', 'form_ga']]
            .to_dict(orient='index')
        )
        snapshots[pred_date] = latest
    return snapshots


form_snapshots = compute_form_snapshots(live_sorted, prediction_dates)
print('Form snapshots computed.')

# Show effect: compare pre-tournament vs post-first-match form for teams that already played
if len(played_wc) > 0 and len(prediction_dates) > 1:
    played_teams = sorted(set(played_wc['home_team']) | set(played_wc['away_team']))
    d0, d1 = prediction_dates[0], prediction_dates[1]
    print(f'\nForm shift for matchday-1 teams  (before {str(d0)[:10]} → after {str(d1)[:10]}):')
    for team in played_teams:
        f0 = form_snapshots[d0].get(team, {})
        f1 = form_snapshots[d1].get(team, {})
        print(f'  {team:<25}  wr: {f0.get("form_wr", float("nan")):.2f} → {f1.get("form_wr", float("nan")):.2f}  '
              f'gf: {f0.get("form_gf", float("nan")):.2f} → {f1.get("form_gf", float("nan")):.2f}')

Form snapshots computed.

Form shift for matchday-1 teams  (before 2026-06-12 → after 2026-06-13):
  Czech Republic             wr: 0.40 → 0.40  gf: 2.40 → 2.40
  Mexico                     wr: 0.80 → 0.80  gf: 2.20 → 2.20
  South Africa               wr: 0.20 → 0.20  gf: 0.60 → 0.60
  South Korea                wr: 0.60 → 0.60  gf: 1.60 → 1.60


In [6]:
def apply_form_features(df, snapshots):
    df = df.copy()
    for col_prefix, team_col in [('home', 'home_team'), ('away', 'away_team')]:
        df[f'{col_prefix}_form_wr'] = df.apply(
            lambda r: snapshots[r['date']].get(r[team_col], {}).get('form_wr', np.nan), axis=1
        )
        df[f'{col_prefix}_form_gf'] = df.apply(
            lambda r: snapshots[r['date']].get(r[team_col], {}).get('form_gf', np.nan), axis=1
        )
        df[f'{col_prefix}_form_ga'] = df.apply(
            lambda r: snapshots[r['date']].get(r[team_col], {}).get('form_ga', np.nan), axis=1
        )
    return df

unplayed_wc = apply_form_features(unplayed_wc, form_snapshots)

# Add ELO and match-context features (static — pre-tournament snapshot)
elo_snap = pd.read_parquet(INTERIM / 'elo_wc2026.parquet')[['team', 'rating']]
unplayed_wc = unplayed_wc.merge(elo_snap.rename(columns={'team': 'home_team', 'rating': 'home_elo'}), on='home_team', how='left')
unplayed_wc = unplayed_wc.merge(elo_snap.rename(columns={'team': 'away_team', 'rating': 'away_elo'}), on='away_team', how='left')
unplayed_wc['elo_diff']          = unplayed_wc['home_elo'] - unplayed_wc['away_elo']
unplayed_wc['is_neutral']        = 1
unplayed_wc['tournament_weight'] = 3.0

print('Features assembled. NaN check on model inputs:')
nan_counts = unplayed_wc[HOME_FEATURES].isna().sum()
print(nan_counts[nan_counts > 0].to_string() if nan_counts.any() else '  None.')

Features assembled. NaN check on model inputs:
  None.


## 5. Train hybrid GLM on full training set

In [7]:
def build_X(df, features):
    X = df[features].copy().fillna(df[features].median())
    return X.values

X_home = build_X(train, HOME_FEATURES)
X_away = build_X(train, AWAY_FEATURES)
X_tr   = np.vstack([X_home, X_away])
y_tr   = np.concatenate([train['home_score'].values, train['away_score'].values])

pipe = Pipeline([
    ('scaler',  StandardScaler()),
    ('poisson', PoissonRegressor(alpha=0.1, max_iter=300)),
])
pipe.fit(X_tr, y_tr)
print(f'Model trained on {len(X_tr)} rows ({len(train)} matches × 2 sides).')
print(f'Converged: {pipe.named_steps["poisson"].n_iter_} iterations')

Model trained on 22212 rows (11106 matches × 2 sides).
Converged: 15 iterations


## 6. Generate predictions

In [8]:
@lru_cache(maxsize=50000)
def best_score_cached(lh_r, la_r):
    """(h, a) maximising expected Sporza pts given Poisson(lh_r) × Poisson(la_r)."""
    goals  = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint  = np.outer(ph_pmf, pa_pmf)

    def expected_pts(pred_h, pred_a):
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph in range(MAX_GOALS + 1):
        for pa in range(MAX_GOALS + 1):
            ep = expected_pts(ph, pa)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph, pa
    return best_h, best_a, round(best_pts, 4)


def build_X(df, features):
    X = df[features].copy().fillna(df[features].median())
    return X.values

lh = pipe.predict(build_X(unplayed_wc, HOME_FEATURES))
la = pipe.predict(build_X(unplayed_wc, AWAY_FEATURES))

preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]

predictions = unplayed_wc[['date', 'home_team', 'away_team', 'group', 'round']].rename(columns={'round': 'tournament'}).copy()
predictions['lambda_home'] = lh.round(3)
predictions['lambda_away'] = la.round(3)
predictions['pred_home']   = [p[0] for p in preds]
predictions['pred_away']   = [p[1] for p in preds]
predictions['exp_pts']     = [p[2] for p in preds]

print(f'Generated predictions for {len(predictions)} matches.')
print(f'Mean expected Sporza pts: {predictions["exp_pts"].mean():.3f}')

Generated predictions for 70 matches.
Mean expected Sporza pts: 4.203


In [9]:
print(predictions[['date', 'home_team', 'away_team', 'pred_home', 'pred_away',
                    'lambda_home', 'lambda_away', 'exp_pts']].to_string(index=False))

      date            home_team            away_team  pred_home  pred_away  lambda_home  lambda_away  exp_pts
2026-06-18       Czech Republic         South Africa          1          0        1.592        0.914   4.0252
2026-06-18               Mexico          South Korea          1          0        1.572        0.879   4.0554
2026-06-24       Czech Republic               Mexico          0          1        0.864        1.645   4.1458
2026-06-24         South Africa          South Korea          0          1        0.842        1.460   4.0210
2026-06-12               Canada Bosnia & Herzegovina          1          0        1.640        0.744   4.3415
2026-06-13                Qatar          Switzerland          0          1        0.584        2.063   4.8341
2026-06-18          Switzerland Bosnia & Herzegovina          1          0        1.851        0.754   4.4679
2026-06-18               Canada                Qatar          2          0        2.341        0.547   5.0159
2026-06-24

## 7. Prediction distribution analysis

In [10]:
score_counts = predictions.groupby(['pred_home', 'pred_away']).size().reset_index(name='count').sort_values('count', ascending=False)
print('Most common predicted scorelines:')
print(score_counts.head(10).to_string(index=False))

print(f'\nLambda stats:')
print(f'  lambda_home: mean={lh.mean():.3f}  min={lh.min():.3f}  max={lh.max():.3f}')
print(f'  lambda_away: mean={la.mean():.3f}  min={la.min():.3f}  max={la.max():.3f}')

win  = (predictions['pred_home'] > predictions['pred_away']).sum()
draw = (predictions['pred_home'] == predictions['pred_away']).sum()
loss = (predictions['pred_home'] < predictions['pred_away']).sum()
print(f'\nOutcome distribution: home win {win}  draw {draw}  away win {loss}  (total {len(predictions)})')

Most common predicted scorelines:
 pred_home  pred_away  count
         1          0     33
         0          1     27
         2          0      6
         3          0      3
         0          2      1

Lambda stats:
  lambda_home: mean=1.526  min=0.553  max=3.518
  lambda_away: mean=1.120  min=0.470  max=2.662

Outcome distribution: home win 42  draw 0  away win 28  (total 70)


## 8. Save predictions

In [11]:
out_path = OUT_DIR / 'wc2026_predictions.csv'
predictions.to_csv(out_path, index=False)
print(f'Saved {len(predictions)} predictions → {out_path.resolve()}')

Saved 70 predictions → /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/predictions/wc2026_predictions.csv
